In [2]:
import ee
import geemap

In [3]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-cmcarbajal')

# Create a map
Map = geemap.Map()

In [4]:
# 1. AOI
aoi = ee.Geometry.Rectangle([-80.14047121, -6.69765857, -79.17709654, -5.94437422])

# 2. Cloud / cirrus mask + scale
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return (image.updateMask(mask)
                 .divide(10000)  # scale to 0–1
                 .copyProperties(image, image.propertyNames()))

In [5]:
# 3. Collection
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate('2023-08-01', '2023-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
        .filterBounds(aoi)
        .map(mask_s2_clouds))

# 4. Median composite
s2_median = s2.median()

In [6]:
# 5. Visualization
rgb_vis = {'min': 0.0, 'max': 0.3, 'bands': ['B4', 'B3', 'B2']}

Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(s2_median.clip(aoi), rgb_vis, 'Sentinel-2 Median')
Map

Map(center=[-6.321146530659889, -79.65878387499988], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# 6. Select export bands
export_bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
export_img = s2_median.select(export_bands).toFloat().clip(aoi)

In [11]:
# 6a. Function to add multiple vegetation indices to a Sentinel-2 image (expects reflectance scaled 0-1)
def add_veg_indices(image):
    # Band aliases for readability
    B2 = image.select('B2')   # Blue
    B3 = image.select('B3')   # Green
    B4 = image.select('B4')   # Red
    B5 = image.select('B5')   # Red Edge 1
    B6 = image.select('B6')   # Red Edge 2
    B7 = image.select('B7')   # Red Edge 3
    B8 = image.select('B8')   # NIR
    B8A = image.select('B8A') # Red Edge 4
    B11 = image.select('B11') # SWIR1
    B12 = image.select('B12') # SWIR2
    
    # Helper to avoid division by zero
    def safe_div(numerator, denominator):
        return numerator.divide(denominator.where(denominator.eq(0), 1))
    
    L = 0.5  # Soil adjustment factor for SAVI
    # Compute indices
    NDVI   = safe_div(B8.subtract(B4), B8.add(B4)).rename('NDVI')
    EVI    = (B8.subtract(B4).multiply(2.5).divide(B8.add(B4.multiply(6)).subtract(B2.multiply(7.5)).add(1))).rename('EVI')
    SAVI   = (B8.subtract(B4).multiply(1+L).divide(B8.add(B4).add(L))).rename('SAVI')
    MSAVI2 = ((B8.multiply(2).add(1).subtract((B8.multiply(2).add(1)).pow(2).subtract(B8.subtract(B4).multiply(8)).sqrt())).divide(2)).rename('MSAVI2')
    NDWI   = safe_div(B3.subtract(B8), B3.add(B8)).rename('NDWI')  # McFeeters (water)
    MNDWI  = safe_div(B3.subtract(B11), B3.add(B11)).rename('MNDWI')
    NBR    = safe_div(B8.subtract(B12), B8.add(B12)).rename('NBR')
    NDRE   = safe_div(B8.subtract(B5), B8.add(B5)).rename('NDRE')
    GNDVI  = safe_div(B8.subtract(B3), B8.add(B3)).rename('GNDVI')
    GCI    = B8.divide(B3).subtract(1).rename('GCI')
    MCARI  = ((B5.subtract(B4)).subtract((B5.subtract(B3)).multiply(0.2)).multiply(B5.divide(B4.where(B4.eq(0),1)))).rename('MCARI')
    
    # Concatenate and cast to Float32 to avoid mixed Float32/Float64 export errors
    indices = ee.Image.cat([NDVI, EVI, SAVI, MSAVI2, NDWI, MNDWI, NBR, NDRE, GNDVI, GCI, MCARI]).toFloat()
    return image.addBands(indices).toFloat()

In [12]:
# 6b. Apply vegetation index function and visualize one (e.g., NDVI)
s2_with_indices = add_veg_indices(s2_median)
ndvi = s2_with_indices.select('NDVI')
Map.addLayer(ndvi, {'min': 0, 'max': 1, 'palette': ['brown','yellow','green']}, 'NDVI')
Map

Map(bottom=68025.0, center=[-6.001264710802936, -79.37484741210938], controls=(WidgetControl(options=['positio…

In [13]:
# 7. Export to Drive (export full set: spectral bands + vegetation indices)
indices = ['NDVI','EVI','SAVI','MSAVI2','NDWI','MNDWI','NBR','NDRE','GNDVI','GCI','MCARI']
export_all = s2_with_indices.select(export_bands + indices).toFloat().clip(aoi)

task = ee.batch.Export.image.toDrive(
    image=export_all,
    description='S2_Motupe_VegIdx',
    folder='INIA_GEE',
    fileNamePrefix='S2_Motupe_VegIdx_UTM',
    region=aoi,
    scale=10,
    crs='EPSG:32717',
    maxPixels=1e13
)

print('Starting export with bands:', export_bands + indices)
task.start()

Starting export with bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'EVI', 'SAVI', 'MSAVI2', 'NDWI', 'MNDWI', 'NBR', 'NDRE', 'GNDVI', 'GCI', 'MCARI']


In [14]:
# 8. Monitor the task status
task.status()
# Wait for the task to complete
import time
while task.active():
    print('Polling for task (id: {}).'.format(task.id))
    time.sleep(40)  
print('Task completed:', task.status())

Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id: OLREV7NKVTLME5TGJ4ZUFBM6).
Polling for task (id